# RG3 가스: 조건부 정상 순위 앙상블

RG3 RH 정상601/가스19만 사용. 절대 운전상태6 + 과거 흔들림4, 총10개를 다른9개가 비슷한 학습 정상25/75개의 조건부 순위로 변환. 해당 변수는 이웃 매칭 기준에서 제외한다. 학습 정상 자기 자신의 값도 비교 이웃에서 제외한다. 이상 점수가 아닌 학습용 순위 표현이며 물리적 가스 원인 설명을 보장하지 않는다.

RF/ET 두 시드 평균 + std4 전문가와 순위 전문가 평균의 최대 점수 앙상블. 모든 외부 평가 구간은 결측·척도·정상 이웃·분류·임계값 학습에서 제외. 모델/임계값 내부 구간 선택. 이미 본 자료의 중첩 구간 검증이며 새 독립 검증 아님. CN7 미사용.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19
inputs=['Max_Injection_Speed','Average_Screw_RPM','Barrel_Temperature_2','Barrel_Temperature_5','Plasticizing_Position','Filling_Time']
targets=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Cushion_Position','Plasticizing_Time']
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
from sklearn.neighbors import NearestNeighbors
memo={};audit=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 normal=ref[y[ref]==0]
 state=d[['Hopper_Temperature','Barrel_Temperature_2','Barrel_Temperature_5','Cushion_Position','Plasticizing_Time']].copy()
 state['pressure_switch_gap']=d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(state.iloc[normal])
 tr=im.transform(state.iloc[ref]);te=im.transform(state.iloc[q])
 si=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[normal][stdcols])
 st=si.transform(d.iloc[ref][stdcols]);sq=si.transform(d.iloc[q][stdcols])
 base_tr=np.c_[tr,st];base_q=np.c_[te,sq]
 norm=base_tr[y[ref]==0];normal_ids=ref[y[ref]==0]
 sc=RobustScaler().fit(norm)
 context_tr=np.clip(sc.transform(base_tr),-8,8);context_q=np.clip(sc.transform(base_q),-8,8)
 context_norm=context_tr[y[ref]==0]
 rank_views=[]
 for neighbors in [25,75]:
  k=min(neighbors,len(norm)-1)
  ranked=[]
  for vals,contexts,ids in [(base_tr,context_tr,ref),(base_q,context_q,q)]:
   z=np.empty_like(vals)
   for col in range(vals.shape[1]):
    other=np.arange(vals.shape[1])!=col
    nn=NearestNeighbors(n_neighbors=k+1).fit(context_norm[:,other])
    idx=nn.kneighbors(contexts[:,other],return_distance=False)
    for row in range(len(vals)):
     candidates=idx[row][normal_ids[idx[row]]!=ids[row]][:k]
     peers=norm[candidates,col]
     z[row,col]=(1+np.sum(peers<vals[row,col])+.5*np.sum(peers==vals[row,col]))/(len(peers)+2)
   ranked.append(z)
  rank_views.append(tuple(ranked))
 features=[(st,sq)]+rank_views
 scores=[]
 for ft,fq in features:
  vals=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    if len(np.unique(y[ref]))<2:vals.append(np.zeros(len(q)));continue
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
    vals.append(m.fit(ft,y[ref]).predict_proba(fq)[:,1])
  scores.append(np.mean(vals,axis=0))
 memo[key]=np.column_stack([scores[0],scores[1],scores[2],np.maximum(scores[0],(scores[1]+scores[2])/2)])
 audit.append(dict(train=len(ref),query=len(q),normal_imputer_training=len(normal),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['std4_reference','conditional_rank25','conditional_rank75','std4_conditional_rank_ensemble']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   pp=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],pp,labels=[0,1]).ravel()
   rec=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((rec,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda x:x[:3])
 choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0],inner_F1=chosen[1]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished RG3 conditional relationship block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(g.actual,g.score) if g.score.notna().all() else np.nan))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_conditional_rank_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))




Finished RG3 conditional relationship block 1


Finished RG3 conditional relationship block 2


Finished RG3 conditional relationship block 3


Finished RG3 conditional relationship block 4


Finished RG3 conditional relationship block 5


Finished RG3 conditional relationship block 6


Finished RG3 conditional relationship block 7


                         model  budget  TP  FP  FN  TN   recall       F1       AP
            conditional_rank25    0.02   1  32  18 569 0.052632 0.038462 0.045964
            conditional_rank25    0.05   2  48  17 553 0.105263 0.057971 0.045964
            conditional_rank25    0.10   5  74  14 527 0.263158 0.102041 0.045964
            conditional_rank75    0.02   3  19  16 582 0.157895 0.146341 0.077190
            conditional_rank75    0.05   5  48  14 553 0.263158 0.138889 0.077190
            conditional_rank75    0.10   8  61  11 540 0.421053 0.181818 0.077190
   inner_recall_first_selected    0.00   3  65  16 536 0.157895 0.068966      NaN
std4_conditional_rank_ensemble    0.02   3  26  16 575 0.157895 0.125000 0.061512
std4_conditional_rank_ensemble    0.05   3  34  16 567 0.157895 0.107143 0.061512
std4_conditional_rank_ensemble    0.10   4  67  15 534 0.210526 0.088889 0.061512
                std4_reference    0.02   3  26  16 575 0.157895 0.125000 0.061578
                